# Speech Emotion Recognition Dataset Preparation

Notebook ini bertujuan untuk mempersiapkan dataset audio sebelum proses pelatihan model CNN.

Tahapan yang dilakukan meliputi:

1. Membaca seluruh file audio dari dataset
2. Mengubah audio menjadi representasi Mel Spectrogram
3. Melakukan padding atau trimming agar ukuran data seragam
4. Melakukan normalisasi fitur
5. Menyimpan dataset ke dalam file NumPy (`X_dataset.npy` dan `y_dataset.npy`)

In [1]:
# ==========================================================
# IMPORT LIBRARY
# ==========================================================

import os
import librosa
import numpy as np

from tqdm import tqdm

In [2]:
# ==========================================================
# CONFIGURATION
# ==========================================================

# Dataset Folder
DATASET_PATH = "Dataset_Final_Indonesia"

# Audio
SAMPLE_RATE = 16000

# Mel Spectrogram
N_MELS = 128

# CNN Input
MAX_LENGTH = 215

In [3]:
# ==========================================================
# EMOTION LABEL
# ==========================================================

emotion_map = {

    "01_Anger": 0,

    "02_Sadness": 1,

    "03_Neutral": 2,

    "04_Happiness": 3

}

In [4]:
# ==========================================================
# CHECK DATASET
# ==========================================================

print("Dataset Folder :")
print(DATASET_PATH)

print("\nEmotion Classes :")

for emotion in emotion_map.keys():

    folder = os.path.join(
        DATASET_PATH,
        emotion
    )

    total = len(os.listdir(folder))

    print(f"{emotion} : {total} files")

Dataset Folder :
Dataset_Final_Indonesia

Emotion Classes :
01_Anger : 400 files
02_Sadness : 400 files
03_Neutral : 461 files
04_Happiness : 460 files


In [5]:
# ==========================================================
# LOAD AUDIO
# ==========================================================

def load_audio(audio_path):

    audio, sr = librosa.load(
        audio_path,
        sr=SAMPLE_RATE
    )

    return audio, sr

In [6]:
# ==========================================================
# CREATE MEL SPECTROGRAM
# ==========================================================

def create_mel_spectrogram(audio, sr):

    mel = librosa.feature.melspectrogram(
        y=audio,
        sr=sr,
        n_mels=N_MELS
    )

    mel_db = librosa.power_to_db(
        mel,
        ref=np.max
    )

    return mel_db

In [7]:
# ==========================================================
# PAD OR TRIM MEL SPECTROGRAM
# ==========================================================

def pad_or_trim(mel_db):

    pad_width = MAX_LENGTH - mel_db.shape[1]

    if pad_width > 0:

        mel_db = np.pad(
            mel_db,
            ((0, 0), (0, pad_width)),
            mode="constant"
        )

    else:

        mel_db = mel_db[:, :MAX_LENGTH]

    return mel_db

In [8]:
# ==========================================================
# NORMALIZE FEATURE
# ==========================================================

def normalize_feature(feature):

    feature = feature.astype("float32")

    denominator = feature.max() - feature.min()

    if denominator == 0:

        return np.zeros_like(feature)

    feature = (
        feature - feature.min()
    ) / denominator

    return feature

In [9]:
# ==========================================================
# BUILD DATASET
# ==========================================================

X = []

y = []

processed_files = 0

failed_files = []

for emotion, label in emotion_map.items():

    folder = os.path.join(
        DATASET_PATH,
        emotion
    )

    print(f"\nProcessing {emotion}...")

    audio_files = sorted(os.listdir(folder))

    for file_name in tqdm(audio_files):

        try:

            audio_path = os.path.join(
                folder,
                file_name
            )

            # Load Audio
            audio, sr = load_audio(
                audio_path
            )

            # Mel Spectrogram
            mel_db = create_mel_spectrogram(
                audio,
                sr
            )

            # Padding / Trimming
            mel_db = pad_or_trim(
                mel_db
            )

            # Normalization
            mel_db = normalize_feature(
                mel_db
            )

            X.append(
                mel_db
            )

            y.append(
                label
            )
            
            processed_files += 1

        except Exception as e:

            failed_files.append(
                (file_name, str(e))
            )


Processing 01_Anger...


100%|████████████████████████████████████████████████████████████████████████████████| 400/400 [00:16<00:00, 24.93it/s]



Processing 02_Sadness...


100%|████████████████████████████████████████████████████████████████████████████████| 400/400 [00:11<00:00, 35.24it/s]



Processing 03_Neutral...


 90%|███████████████████████████████████████████████████████████████████████▊        | 414/461 [00:11<00:01, 32.61it/s]C:\Users\hafiz\AppData\Local\Temp\ipykernel_19596\3125534355.py:7: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, sr = librosa.load(
C:\Users\hafiz\anaconda3\Lib\site-packages\librosa\core\audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
100%|████████████████████████████████████████████████████████████████████████████████| 461/461 [00:13<00:00, 33.64it/s]



Processing 04_Happiness...


100%|████████████████████████████████████████████████████████████████████████████████| 460/460 [00:12<00:00, 36.93it/s]


In [10]:
# ==========================================================
# CONVERT TO NUMPY ARRAY
# ==========================================================

X = np.array(
    X,
    dtype="float32"
)

X = X[..., np.newaxis]

y = np.array(
    y,
    dtype="int32"
)

print("Dataset successfully created.")

Dataset successfully created.


In [11]:
# ==========================================================
# DATASET INFORMATION
# ==========================================================

print(f"X Shape : {X.shape}")

print(f"y Shape : {y.shape}")

print(f"Processed Files : {processed_files}")

print(f"Failed Files : {len(failed_files)}")

X Shape : (1721, 128, 215, 1)
y Shape : (1721,)
Processed Files : 1721
Failed Files : 0


In [12]:
# ==========================================================
# FAILED FILES
# ==========================================================

if failed_files:

    print("Failed Files:")

    for file_name, error in failed_files:

        print(file_name)

        print(error)

        print("-" * 40)

else:

    print("All audio files processed successfully.")

All audio files processed successfully.


In [13]:
# ==========================================================
# LABEL DISTRIBUTION
# ==========================================================

label_names = {

    0: "Anger",

    1: "Sadness",

    2: "Neutral",

    3: "Happiness"

}

print("Dataset Distribution\n")

unique, counts = np.unique(
    y,
    return_counts=True
)

for label, total in zip(unique, counts):

    print(f"{label_names[label]:<12}: {total}")

Dataset Distribution

Anger       : 400
Sadness     : 400
Neutral     : 461
Happiness   : 460


In [14]:
# ==========================================================
# SAVE DATASET
# ==========================================================

np.save(

    "X_dataset.npy",

    X

)

np.save(

    "y_dataset.npy",

    y

)

print(

    "Dataset saved successfully."

)

Dataset saved successfully.


In [15]:
# ==========================================================
# VERIFY SAVED FILE
# ==========================================================

print("Saved Files\n")

files = [

    "X_dataset.npy",

    "y_dataset.npy"

]

for file in files:

    if os.path.exists(file):

        size = os.path.getsize(file) / (1024 * 1024)

        print(f"{file}  ({size:.2f} MB)")

    else:

        print(f"{file}  Not Found")

Saved Files

X_dataset.npy  (180.67 MB)
y_dataset.npy  (0.01 MB)


In [16]:
# ==========================================================
# SUMMARY
# ==========================================================

print("=" * 50)

print("DATASET PREPARATION COMPLETED")

print("=" * 50)

print(f"Total Samples     : {len(y)}")

print(f"Feature Shape     : {X.shape}")

print(f"Label Shape       : {y.shape}")

print(f"Output Feature    : X_dataset.npy")

print(f"Output Label      : y_dataset.npy")

print("=" * 50)

DATASET PREPARATION COMPLETED
Total Samples     : 1721
Feature Shape     : (1721, 128, 215, 1)
Label Shape       : (1721,)
Output Feature    : X_dataset.npy
Output Label      : y_dataset.npy
